# Kaggle End-to-End (2xT4 with fallback)

This notebook downloads the dataset from the HF Hub, prepares a 50k sample with age prefixes, trains GPT-2, evaluates readability, saves the model, and uploads to your Hugging Face repo.

If 2 GPUs are available, it uses DataParallel. Otherwise it falls back to a single GPU or CPU.

In [1]:
import os
import sys
import subprocess
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))

BASE_DIR = '/kaggle/working'
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw_hf')
PROC_DIR = os.path.join(BASE_DIR, 'data', 'processed')
MODEL_OUT = os.path.join(BASE_DIR, 'model_output')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(MODEL_OUT, exist_ok=True)

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [2]:
REPO_URL = 'https://github.com/khedimyoucef/NLP-MINI-PROJECT.git'
REPO_DIR = 'NLP-MINI-PROJECT'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL], check=True)
sys.path.insert(0, REPO_DIR)
print('Repo ready:', REPO_DIR)

Cloning into 'NLP-MINI-PROJECT'...


Repo ready: NLP-MINI-PROJECT


In [3]:
!pip -q install -r NLP-MINI-PROJECT/requirements.txt
!pip -q install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 55.3 MB/s eta 0:00:00:00:01


In [4]:
import os
from huggingface_hub import login

token = os.environ.get('HF_TOKEN')
if token:
    login(token=token)
    print('Logged in with HF_TOKEN')
else:
    print('HF_TOKEN not found; you can still run, but upload will ask for login')

HF_TOKEN not found; you can still run, but upload will ask for login


In [5]:
from datasets import load_dataset

ds = load_dataset("ajibawa-2023/Children-Stories-Collection", split='train', cache_dir=os.path.join(BASE_DIR, 'hf_cache'))
print('Columns:', ds.column_names)
print('Rows:', len(ds))
ds.save_to_disk(RAW_DIR)
print('Saved raw dataset to', RAW_DIR)
print(ds[0])

README.md:   0%|          | 0.00/625 [00:00<?, ?B/s]

Children-Stories-0-Final.json:   0%|          | 0.00/344M [00:00<?, ?B/s]

Children-Stories-1-Final.json:   0%|          | 0.00/346M [00:00<?, ?B/s]

Children-Stories-2-Final.json:   0%|          | 0.00/343M [00:00<?, ?B/s]

Children-Stories-3-Final.json:   0%|          | 0.00/345M [00:00<?, ?B/s]

...

Children-Stories-9-Final.json:   0%|          | 0.00/342M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/896668 [00:00<?, ? examples/s]

Columns: ['text_token_length', 'text', 'prompt']
Rows: 896668


Saving the dataset (0/7 shards):   0%|          | 0/896668 [00:00<?, ? examples/s]

Saved raw dataset to /kaggle/working/data/raw_hf
{'text_token_length': 395, 'text': ' Once upon a time in the land of Policymia, there lived two leaders named Majora and Minoro. Their job was to make sure all the citizens had beautiful parks, clean water, and top-notch schools. But there were so many things to fix! How would they ever decide where to start?\n\nMajora, being the wise leader she was, knew just what to do. She invited her fellow policymakers for a big meeting at the Roundtable of Representatives. There, they discussed the most important problems Policymia faced. This was called identifying "key policy areas." It meant figuring out which topics needed attention first.\n\nNext came assessing support – finding out if everyone agreed on the solutions. Some people thought building more playgrounds was the way to go, while others wanted better libraries. To understand everyone\'s thoughts, Majora used something called \'polling.\' Just like taking a vote, polling helped her see

In [6]:
from datasets import Dataset

PREFIXES = [
    'Level: Age3-4 — Simple words.',
    'Level: Age5-6 — Short sentences.',
    'Level: Age7-8 — Moderate vocabulary.',
    'Level: Age9-10 — Longer sentences.',
    'Level: Age11-12 — Richer vocabulary.'
]

def to_text(example):
    if 'text' in example and example['text']:
        return example['text']
    if 'story' in example and example['story']:
        return example['story']
    prompt = example.get('prompt', '')
    body = example.get('completion', '') or example.get('response', '') or example.get('text', '')
    return (prompt + ' ' + body).strip()

sample_size = 50000
ds_sample = ds.shuffle(seed=42).select(range(min(sample_size, len(ds))))

def add_prefix(example, idx):
    prefix = PREFIXES[idx % len(PREFIXES)]
    text = to_text(example)
    return {'text': f'{prefix} {text}'}

ds_pref = ds_sample.map(add_prefix, with_indices=True, remove_columns=ds_sample.column_names)
out_path = os.path.join(PROC_DIR, 'prefix_stories.json')
ds_pref.to_json(out_path)
print('Saved', out_path)
print(ds_pref[0]['text'][:200])

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/50 [00:00<?, ?ba/s]

Saved /kaggle/working/data/processed/prefix_stories.json
Level: Age3-4 — Simple words.  Once upon a time in a small town named Greenville, lived two best friends, Sam and Alex. They loved spending time outdoors, exploring the beautiful forests and rivers ar


In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

ds_tok = load_dataset('json', data_files=out_path, split='train')

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

ds_tok = ds_tok.map(tokenize, batched=True, remove_columns=['text'])
ds_tok = ds_tok.train_test_split(test_size=0.05, seed=42)
train_ds = ds_tok['train']
val_ds = ds_tok['test']
train_ds.set_format('torch')
val_ds.set_format('torch')
print('Train size:', len(train_ds), 'Val size:', len(val_ds))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train size: 47500 Val size: 2500


In [ ]:
### import math
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device_count = torch.cuda.device_count()

model = AutoModelForCausalLM.from_pretrained('gpt2')
model = model.to(device)
if device_count > 1:
    model = torch.nn.DataParallel(model)

per_device_batch = 4
grad_accum = 4
epochs = 2
lr = 2e-5

train_loader = DataLoader(train_ds, batch_size=per_device_batch, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler() if use_amp else None

print('Training...')
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch['input_ids']
        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(**batch, labels=labels)
            loss = outputs.loss
            if loss.ndim > 0:
                loss = loss.mean()
            loss = loss / grad_accum
        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        if step % grad_accum == 0:
            if scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        total_loss += float(loss.detach().cpu())
        if step % 50 == 0:
            print(f'Epoch {epoch} Step {step} Loss {total_loss/step:.4f}')
    print(f'Epoch {epoch} done, avg loss {total_loss/step:.4f}')

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/tmp/ipykernel_55/150594932.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if use_amp else None
/tmp/ipykernel_55/150594932.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Training...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch 1 Step 50 Loss 0.8410
Epoch 1 Step 100 Loss 0.8067
Epoch 1 Step 150 Loss 0.7828
Epoch 1 Step 200 Loss 0.7647
Epoch 1 Step 250 Loss 0.7501
Epoch 1 Step 300 Loss 0.7399
Epoch 1 Step 350 Loss 0.7302
Epoch 1 Step 400 Loss 0.7229
Epoch 1 Step 450 Loss 0.7163
Epoch 1 Step 500 Loss 0.7105
Epoch 1 Step 550 Loss 0.7065
Epoch 1 Step 600 Loss 0.7028
Epoch 1 Step 650 Loss 0.6994
Epoch 1 Step 700 Loss 0.6954
Epoch 1 Step 750 Loss 0.6925
Epoch 1 Step 800 Loss 0.6895
Epoch 1 Step 850 Loss 0.6870
Epoch 1 Step 900 Loss 0.6846
Epoch 1 Step 950 Loss 0.6825
Epoch 1 Step 1000 Loss 0.6803
Epoch 1 Step 1050 Loss 0.6785
Epoch 1 Step 1100 Loss 0.6765
Epoch 1 Step 1150 Loss 0.6746
Epoch 1 Step 1200 Loss 0.6727
Epoch 1 Step 1250 Loss 0.6710
Epoch 1 Step 1300 Loss 0.6697
Epoch 1 Step 1350 Loss 0.6682
Epoch 1 Step 1400 Loss 0.6668
Epoch 1 Step 1450 Loss 0.6654
Epoch 1 Step 1500 Loss 0.6642
Epoch 1 Step 1550 Loss 0.6633
Epoch 1 Step 1600 Loss 0.6622
Epoch 1 Step 1650 Loss 0.6610
Epoch 1 Step 1700 Loss 0.6597


In [9]:
# Evaluation: perplexity
from torch.utils.data import DataLoader
model.eval()
val_loader = DataLoader(val_ds, batch_size=8)
losses = []
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch['input_ids']
        outputs = model(**batch, labels=labels)
        loss = outputs.loss
        if loss.ndim > 0:
            loss = loss.mean()
        losses.append(float(loss.detach().cpu()))
avg_loss = sum(losses) / len(losses)
ppl = math.exp(avg_loss)
print('Validation loss:', avg_loss)
print('Perplexity:', ppl)

Validation loss: 2.1008748040793424
Perplexity: 8.173316836943282


In [10]:
# Readability check across age prefixes
import textstat
gen_model = model.module if hasattr(model, 'module') else model
theme = 'a dragon afraid of fire'
for prefix in PREFIXES:
    prompt = prefix + ' ' + theme
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = gen_model.generate(**inputs, max_new_tokens=120, do_sample=True, temperature=0.8)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(prefix, 'FK grade:', textstat.flesch_kincaid_grade(text))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Level: Age3-4 — Simple words. FK grade: 4.58407843137255


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Level: Age5-6 — Short sentences. FK grade: 5.131721094439541


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Level: Age7-8 — Moderate vocabulary. FK grade: 6.500925925925927


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Level: Age9-10 — Longer sentences. FK grade: 4.796830188679245
Level: Age11-12 — Richer vocabulary. FK grade: 4.319144736842109


In [11]:
# Save model and tokenizer
gen_model = model.module if hasattr(model, 'module') else model
gen_model.save_pretrained(MODEL_OUT)
tokenizer.save_pretrained(MODEL_OUT)
print('Saved model to', MODEL_OUT)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to /kaggle/working/model_output


In [15]:
from huggingface_hub import login, whoami
login()  # paste token with write access
print(whoami())

{'type': 'user', 'id': '68d1772a0ea4a50a49a0588e', 'name': 'khedim', 'fullname': 'khedim youcef', 'email': 'khedimyoucefdz@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'isPro': False, 'avatarUrl': '/avatars/cdb883bd091c10f2aa84ccc29a74c30c.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'Kaggle Access Token', 'role': 'write', 'createdAt': '2026-04-24T17:11:29.270Z'}}}


In [16]:
# Upload to Hugging Face Hub
from huggingface_hub import upload_folder
upload_folder(folder_path=MODEL_OUT, repo_id='khedim/NLP-MINI-PROJECT', repo_type='model')
print('Uploaded to Hugging Face: khedim/NLP-MINI-PROJECT')

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded to Hugging Face: khedim/NLP-MINI-PROJECT
